# SpaceRL — Versión 1: Q-learning en Taxi-v3

> **Proyecto:** SpaceRL  
> **Versión:** 1 — Entorno estándar de Gymnasium  
> **Algoritmo:** Q-learning tabular con política epsilon-greedy  
> **Autores:** Álvaro · Marta · Paula  

---

## Objetivo

Demostrar que comprendemos los fundamentos del aprendizaje por refuerzo usando un entorno clásico y controlado:

| Concepto | Implementación |
|---|---|
| Entorno | `Taxi-v3` (Gymnasium) |
| Algoritmo | Q-learning tabular |
| Política | ε-greedy con decaimiento |
| Tabla Q | `(500 estados, 6 acciones)` |
| Métricas | Recompensa, pasos, TD error, epsilon |

---

## 0. Imports y configuración

In [ ]:
import sys, os
# Make sure the v1/ root is in the Python path when running from notebooks/
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# SpaceRL modules
from config import (
    ENV_NAME, N_EPISODES, MAX_STEPS, LEARNING_RATE, DISCOUNT_FACTOR,
    EPSILON_START, EPSILON_MIN, EPSILON_DECAY, LOG_EVERY,
    N_EVAL_EPS, QTABLE_PATH, METRICS_NPZ_PATH, METRICS_CSV_PATH,
    EVAL_CSV_PATH, FIGURE_PATH, WINDOW_SIZE,
)
from agent import QLearningAgent
from environment import make_env, env_info
from training import run_training
from evaluation.eval_runner import evaluate_agent
from evaluation.plot_metrics import build_figure, load_metrics_df, moving_average
from utils import (
    save_training_metrics, save_evaluation_metrics,
    training_summary, evaluation_summary,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print(' Imports OK')

## 1. Hiperparámetros

Todos los valores se centralizan en `config.py`. Aquí los mostramos para tenerlos visibles en el notebook.

In [ ]:
hyperparams = pd.DataFrame([
    {'Parámetro': 'N_EPISODES',      'Valor': N_EPISODES,      'Descripción': 'Episodios de entrenamiento'},
    {'Parámetro': 'MAX_STEPS',       'Valor': MAX_STEPS,       'Descripción': 'Pasos máximos por episodio'},
    {'Parámetro': 'LEARNING_RATE',   'Valor': LEARNING_RATE,   'Descripción': 'Alpha — velocidad de actualización Q'},
    {'Parámetro': 'DISCOUNT_FACTOR', 'Valor': DISCOUNT_FACTOR, 'Descripción': 'Gamma — importancia de recompensas futuras'},
    {'Parámetro': 'EPSILON_START',   'Valor': EPSILON_START,   'Descripción': 'Exploración inicial'},
    {'Parámetro': 'EPSILON_MIN',     'Valor': EPSILON_MIN,     'Descripción': 'Exploración mínima'},
    {'Parámetro': 'EPSILON_DECAY',   'Valor': EPSILON_DECAY,   'Descripción': 'Factor de decaimiento por episodio'},
])
hyperparams.style.set_caption('Hiperparámetros del experimento')

## 2. Entorno

**Taxi-v3** es un entorno de cuadrícula 5×5 con:
- **500 estados**: posición del taxi (25) × posición del pasajero (5) × destino (4)
- **6 acciones**: Norte, Sur, Este, Oeste, Recoger pasajero, Dejar pasajero
- **Recompensas**: -1 por paso, -10 por acción ilegal, +20 por entrega correcta

In [ ]:
env = make_env()
info = env_info(env)

env_df = pd.DataFrame([info])
display(env_df)

# Show one random initial observation
obs, _ = env.reset(seed=42)
print(f'\nEstado inicial (ejemplo): {obs}')
print(f'Espacio de acciones: {env.action_space}')
env.close()

## 3. Entrenamiento

Ejecutar la siguiente celda entrena al agente y guarda los resultados en `results/`.

> Si ya tienes `results/metrics/metrics.csv`, puedes **saltar esta celda** y cargar los resultados guardados en la sección 4.

In [ ]:
env   = make_env()
agent = QLearningAgent(env)

print(f'Entrenando durante {N_EPISODES:,} episodios...\n')
rewards, steps, td_errors, epsilon_history = run_training(env, agent)
env.close()

# Save Q-table
agent.save(QTABLE_PATH)

# Save metrics (npz + CSV)
df_train = save_training_metrics(
    rewards, steps, td_errors, epsilon_history,
    npz_path=METRICS_NPZ_PATH,
    csv_path=METRICS_CSV_PATH,
)

print('\nEntrenamiento completado.')

## 4. Análisis de métricas de entrenamiento

Cargamos el CSV con pandas para explorar los datos antes de graficarlos.

In [ ]:
df_train = load_metrics_df(prefer_csv=True)
print(f'Episodios totales: {len(df_train):,}')
df_train.head(10)

In [ ]:
# Descriptive statistics
df_train[['reward', 'steps', 'td_error', 'epsilon']].describe().round(4)

In [ ]:
# Summary of the last 500 episodes
summary = training_summary(df_train, last_n=500)
display(summary.style.set_caption('Resumen: últimos 500 episodios'))

In [ ]:
# Convergence check: reward quartile progression across training
df_train['block'] = pd.cut(df_train['episode'], bins=10, labels=False)
convergence = df_train.groupby('block')['reward'].agg(['mean', 'median', 'std']).round(2)
convergence.index = [f'Bloque {i+1}' for i in convergence.index]
convergence.columns = ['Media recompensa', 'Mediana recompensa', 'Desv. típica']
display(convergence.style.set_caption('Progresión del aprendizaje por bloques de episodios'))

## 5. Visualización

Generamos las curvas de aprendizaje (figura de 4 paneles).

In [ ]:
build_figure(df_train, save_path=FIGURE_PATH, show=True)

In [ ]:
# Additional plot: reward distribution in last 1000 episodes
fig, ax = plt.subplots(figsize=(8, 4))
last_1k = df_train.tail(1000)['reward']
ax.hist(last_1k, bins=30, color='#2196F3', edgecolor='white', alpha=0.85)
ax.axvline(last_1k.mean(), color='#FF5722', linewidth=2, linestyle='--', label=f'Media: {last_1k.mean():.1f}')
ax.set_title('Distribución de recompensas — últimos 1000 episodios', fontsize=11, fontweight='bold')
ax.set_xlabel('Recompensa total')
ax.set_ylabel('Frecuencia')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## 6. Evaluación del agente entrenado

Evaluamos con política completamente greedy (ε = 0). Sin exploración.

In [ ]:
# Load agent
env_eval = make_env()
agent_eval = QLearningAgent(env_eval)
env_eval.close()
agent_eval.load(QTABLE_PATH)

# Run evaluation
eval_rewards, eval_steps, eval_successes = evaluate_agent(agent_eval, n_episodes=N_EVAL_EPS)

# Save to CSV
df_eval = save_evaluation_metrics(eval_rewards, eval_steps, eval_successes, csv_path=EVAL_CSV_PATH)

print('\n Evaluación completada.')

In [ ]:
df_eval.head(10)

In [ ]:
summary_eval = evaluation_summary(df_eval)
display(summary_eval.style.set_caption('Resumen de evaluación (política greedy pura)'))

In [ ]:
# Evaluation reward distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_eval['reward'], bins=20, color='#4CAF50', edgecolor='white', alpha=0.85)
axes[0].axvline(df_eval['reward'].mean(), color='#FF5722', linestyle='--', linewidth=2,
                label=f"Media: {df_eval['reward'].mean():.1f}")
axes[0].set_title('Distribución de recompensas (evaluación)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Recompensa total')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.4)

axes[1].hist(df_eval['steps'], bins=20, color='#9C27B0', edgecolor='white', alpha=0.85)
axes[1].axvline(df_eval['steps'].mean(), color='#FF5722', linestyle='--', linewidth=2,
                label=f"Media: {df_eval['steps'].mean():.1f}")
axes[1].set_title('Distribución de pasos (evaluación)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Número de pasos')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.4)

plt.suptitle('SpaceRL v1 — Análisis de evaluación', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Análisis de la política aprendida

Inspeccionamos la Q-table para ver qué ha aprendido el agente.

In [ ]:
qtable = agent_eval.qtable
print(f'Forma de la Q-table: {qtable.shape}')
print(f'Q-values no-nulos: {np.count_nonzero(qtable):,} / {qtable.size:,}')
print(f'Rango de Q-values: [{qtable.min():.3f}, {qtable.max():.3f}]')

In [ ]:
# Action names for Taxi-v3
ACTION_NAMES = ['Sur', 'Norte', 'Este', 'Oeste', 'Recoger', 'Dejar']

# Show Q-values for a few interesting states
sample_states = [0, 100, 200, 300, 400, 499]
rows = []
for s in sample_states:
    row = {'Estado': s, 'Acción óptima': ACTION_NAMES[np.argmax(qtable[s])]}
    for a, name in enumerate(ACTION_NAMES):
        row[name] = round(qtable[s, a], 3)
    rows.append(row)

pd.DataFrame(rows).style.set_caption('Q-values para estados de muestra').highlight_max(
    subset=ACTION_NAMES, axis=1, color='#c8e6c9'
)

In [ ]:
# Heatmap: action frequency across all states
best_actions = np.argmax(qtable, axis=1)
action_counts = pd.Series(best_actions).value_counts().sort_index()
action_counts.index = [ACTION_NAMES[i] for i in action_counts.index]

fig, ax = plt.subplots(figsize=(8, 4))
action_counts.plot(kind='bar', ax=ax, color='#2196F3', edgecolor='white', alpha=0.85)
ax.set_title('Acción preferida (greedy) por estado', fontsize=11, fontweight='bold')
ax.set_xlabel('Acción')
ax.set_ylabel('Número de estados')
ax.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 8. Resumen final

Consolidamos los resultados clave del experimento.

In [ ]:
print('=' * 55)
print('  SpaceRL v1 — Taxi-v3 Q-learning  |  Resultados')
print('=' * 55)

train_s = training_summary(df_train, last_n=500)
eval_s  = evaluation_summary(df_eval)

print('\n Entrenamiento (últimos 500 episodios):')
print(f"  Recompensa media : {train_s['mean_reward'].values[0]:.2f}")
print(f"  Pasos medios     : {train_s['mean_steps'].values[0]:.1f}")
print(f"  TD error medio   : {train_s['mean_td_error'].values[0]:.5f}")
print(f"  Epsilon final    : {train_s['final_epsilon'].values[0]:.4f}")

print('\n Evaluación (política greedy, 100 episodios):')
print(f"  Recompensa media : {eval_s['mean_reward'].values[0]:.2f}  ±{eval_s['std_reward'].values[0]:.2f}")
print(f"  Pasos medios     : {eval_s['mean_steps'].values[0]:.1f}")
print(f"  Tasa de éxito    : {eval_s['success_rate'].values[0]:.1f}%")
print('=' * 55)

## 9. Archivos generados

| Archivo | Ruta | Contenido |
|---|---|---|
| Q-table | `results/models/qtable.npy` | Tabla Q aprendida (500×6) |
| Métricas (binario) | `results/metrics/metrics.npz` | Numpy archive |
| Métricas (CSV) | `results/metrics/metrics.csv` | Por episodio, legible con pandas |
| Evaluación (CSV) | `results/metrics/evaluation.csv` | Resultados greedy por episodio |
| Figura | `results/figures/training_metrics.png` | Curvas de aprendizaje (4 paneles) |

---

## Conexión con las versiones siguientes

- **Versión 2:** Aplicaremos *wrappers* sobre Taxi-v3 para personalizar recompensas y observaciones, dando un toque espacial sin necesidad de crear un entorno desde cero.
- **Versión 3:** `SpaceMissionEnv` — entorno personalizado de misión espacial con recursos, eventos aleatorios y una política más rica. La estructura modular de este código (agente / entorno / training / evaluation) se reutiliza directamente.
